In [90]:
model = "llama3.2:1B"

#### Task 1: Simple Chain with Retrieval

**Objective:**

Implement a simple RAG chain with ChatOllama, HuggingFaceEmbeddings and Chroma. 

Process: 

1. Retrieve documents from chroma db based on query
2. Invoke chain with retrieved documents as input

**Task Description:**

- load llm model via ollama
- load embedding model via ollama with `ollama pull pull bge-m3` (if not yet done)
- create chroma db client
- create prompt template for summarization
- create simple chain with following steps: retrieved documents, prompt, model, output parser
- create query and perform similarity search with a query
- invoke chain and pass retrieved documents to the chain


**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)
- [Streaming in Langchain](https://python.langchain.com/docs/concepts/streaming/)


In [91]:
from langchain_ollama import ChatOllama

# ADD HERE YOUR CODE
model = "llama3.2:1B"

In [92]:
from langchain_ollama import OllamaEmbeddings

# ADD HERE YOUR CODE
embedding_model = OllamaEmbeddings(model="llama3.2:1B")


In [93]:

from langchain_chroma import Chroma
import chromadb
import chromadb
from chromadb.config import DEFAULT_TENANT, DEFAULT_DATABASE, Settings

client = chromadb.HttpClient(
    host="localhost",
    port=8000,
    ssl=False,
    headers=None,
    settings=Settings(allow_reset=True, anonymized_telemetry=False),
    tenant=DEFAULT_TENANT,
    database=DEFAULT_DATABASE,
)

# Create a collection
# ADD HERE YOUR CODE
collection = client.get_or_create_collection(name="my_collection")


# Create chromadb
# ADD HERE YOUR CODE
vector_db_from_client = Chroma(collection_name="my_collection", embedding_function=embedding_model, client=client)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import WikipediaLoader


prompt = ChatPromptTemplate.from_template(
    "Summarize the main themes in these retrieved docs: {docs}"
)

# Verwende Mock-Dokumente mit page_content Attribut
class MockDocument:
    def __init__(self, content, doc_id=None):
        self.page_content = content
        self.metadata = {}
        self.id = doc_id

docs = [
    MockDocument("""Machine learning is a branch of artificial intelligence that focuses on 
    building applications that learn from data and improve their performance over time without 
    being explicitly programmed. It involves algorithms and statistical models that enable 
    computers to learn from and make decisions or predictions based on data.
    
    Key types include supervised learning (with labeled data), unsupervised learning (pattern discovery),
    and reinforcement learning (learning through interaction). Common applications include image recognition,
    natural language processing, recommendation systems, and predictive analytics.""", doc_id="ml_doc_1"),
]

# Speichere Dokumente in der Chroma Vector DB
vector_db_from_client.add_documents(docs)

# Convert loaded documents into strings by concatenating their content
# and ignoring metadata
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


chain = prompt | ChatOllama(model=model) | StrOutputParser()

result = chain.invoke({"docs": format_docs(docs)})
print(result)

AttributeError: 'MockDocument' object has no attribute 'id'

In [95]:
search_query = "Types of Machine Learning Systems"

# ADD HERE YOUR CODE
# Perform vector search
docs = vector_db_from_client.similarity_search(search_query, k=5)

print(docs)

[]


In [96]:
chain.invoke({"docs": format_docs(docs)})

"However, I need to clarify that I don't see any retrieved documents. Could you please provide me with some sample text or a specific set of documents that you'd like me to summarize? Alternatively, if you're referring to a particular project or task, feel free to share more context and I'll do my best to assist you."

In [97]:
# Simple stream the chain output
for chunk in chain.stream({"docs": format_docs(docs)}):
    print(chunk, end="", flush=True)

However, I need to clarify that you haven't provided any documents for me to summarize. Please share the retrieved docs with me, and I'll be happy to help you identify the main themes.

In [98]:
# More complex async event streaming
async for event in chain.astream_events({"docs": format_docs(docs)}, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

However, I need to clarify that I'm a large language model, I don't see any retrieved documents. Can you please provide me with the retrieved documents or the text you'd like me to summarize? Additionally, which specific themes would you like me to highlight? I'll do my best to provide a concise and accurate summary.

#### Task 2: Q&A with RAG

**Objective:**

Implement a Q/A retrieval chain with ChatOllama, HuggingFaceEmbeddings and Chroma

**Task Description:**

- create RAG-Q/A prompt template
- create retriever from vector db client (instead of manually passing in docs, we automatically retrieve them from our vector store based on the user question)
- create simple chain with following steps: retriever, formatting retrieved docs, user question, prompt, model, output parser
- create question for Q/A retrieval chain
- invoke chain and with question

**Useful links:**

- [RAG with Ollama](https://python.langchain.com/v0.2/docs/tutorials/local_rag/)

In [99]:
from langchain_core.runnables import RunnablePassthrough

prompt_template = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.

<context>
{context}
</context>

Answer the following question:

{question}"""

# ADD HERE YOUR CODE
rag_prompt = ChatPromptTemplate.from_template(prompt_template)

# ADD HERE YOUR CODE
retriever = vector_db_from_client.as_retriever(search_kwargs={"k": 5})

# ADD HERE YOUR CODE
qa_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | ChatOllama(model=model)
    | StrOutputParser()
)

In [100]:
qa_rag_chain

{
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000020DBC133690>, search_kwargs={'k': 5})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\nYou are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\n\n<context>\n{context}\n</context>\n\nAnswer the following question:\n\n{question}"), additional_kwargs={})])
| ChatOllama(model='llama3.2:1B')
| StrOutputParser()

In [101]:
question = "What is supervised learning?"

# ADD HERE YOUR CODE
qa_rag_chain.invoke(question)

'Supervised learning is a type of machine learning where an algorithm is trained on labeled data to make predictions or classify new instances based on known relationships between inputs and outputs. The data includes both training examples (where the input and output are provided) and testing examples (where the input and output are not provided but used for evaluation). By analyzing the patterns in the labeled data, a model learns to make accurate predictions or decisions without needing additional supervision during the learning process.'

In [102]:
# More complex async event streaming
async for event in qa_rag_chain.astream_events(question, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where an algorithm is trained on labeled data, meaning it is given correct or desired outputs to learn from, and uses this information to make predictions or decisions. The goal is for the model to accurately classify new, unseen data based on the patterns learned from the existing data. This process involves feeding the training data into the algorithm multiple times, adjusting its parameters to minimize errors, until it reaches a predetermined level of accuracy.

#### Alternative: Using pre-built ConversationalRetrievalChain Class

In [103]:
from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory

In [104]:
retriever = vector_db_from_client.as_retriever()
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [105]:
qa_chain = ConversationalRetrievalChain.from_llm(
    ChatOllama(model=model), retriever=retriever, memory=memory, verbose=False
)

In [106]:
# More complex async event streaming
async for event in qa_chain.astream_events("What is supervised learning?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Supervised learning is a type of machine learning where an algorithm is trained on labeled data, meaning the data is already classified or tagged with the correct output. The goal is to teach the algorithm to predict or classify new, unseen data based on the patterns and relationships learned from the labeled data.

In supervised learning, the following steps typically occur:

1. **Data collection**: A dataset of labeled examples is gathered.
2. **Model training**: An algorithm (e.g., neural network) is trained using the labeled data to learn the underlying patterns and relationships.
3. **Model evaluation**: The trained model is evaluated on a test dataset to assess its performance.

Supervised learning has many applications, including:

* Image classification: identifying objects in images
* Sentiment analysis: determining the sentiment of text data
* Predicting continuous values: forecasting sales or stock prices

Examples of supervised learning algorithms include decision trees, su

In [107]:
# More complex async event streaming
async for event in qa_chain.astream_events("Which algorithms can be used there?", version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event["data"]["chunk"].content, end="", flush=True)

Here is the rephrased follow up question:

What type(s) of machine learning algorithms are typically used in supervised learning?I don't know. I don't have enough context to provide a specific list of supervised learning machine learning algorithms. Can you provide more information or clarify your question?